# 🔤 Assignment #3.2 — Word Sense and Paraphrasing with WordNet

## 📋 Overview

This notebook performs automated text analysis and paraphrasing using WordNet, a lexical database for English.

### 🎯 What This Does

For each sentence in your input text, the system will:

1. **🔍 Find Common Themes** — Identify the highest-level concepts that tie the sentence together
2. **🏷️ Assign Word Meanings** — Choose the most appropriate dictionary definition for each content word
3. **✏️ Generate Paraphrases** — Rewrite the sentence with 3-5 synonym replacements while preserving meaning

## 📦 Required NLTK Resources

Before running, download these resources:

```python
nltk.download('punkt')              # Sentence tokenizer
nltk.download('averaged_perceptron_tagger')  # POS tagger
nltk.download('wordnet')            # WordNet database
nltk.download('omw-1.4')            # Multilingual wordnet
nltk.download('stopwords')          # Common stopwords
```


In [43]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk import pos_tag
from nltk.corpus import wordnet as wn, stopwords
import string
import re
from collections import Counter

# If running for the first time, uncomment these:
# nltk.download('punkt')
# nltk.download('averaged_perceptron_tagger')
# nltk.download('wordnet')
# nltk.download('omw-1.4')
# nltk.download('stopwords')

# Stopwords and punctuation sets for filtering
stop_en = set(stopwords.words('english'))
punct = set(string.punctuation)



## 🛠️ Part 1: POS Tag Repair and Mapping

### Why Do We Need This?

Natural language processing tools aren't perfect. The built-in part-of-speech (POS) tagger in NLTK makes mistakes:

| Common Mistakes | Examples |
|----------------|----------|
| **Nouns tagged as verbs** | "system" → VB instead of NN |
| **Verbs tagged as nouns** | "happens" → NNS instead of VBZ |
| **Wrong adjective forms** | Superlatives (JJS) used for regular nouns |
| **Adverbs as nouns** | "quickly" → NN instead of RB |

### Why It Matters

WordNet requires accurate POS tags to find the right word senses:
- ❌ Wrong POS → Wrong definition → Wrong synonym
- ✅ Correct POS → Correct definition → Good synonym

### What We Built

#### `penn_to_wn_pos(tag)`
Maps Penn Treebank tags (NN, VBZ, JJ, RB) to WordNet tags (n, v, a, r).

```
NN  → 'n' (noun)
VBZ → 'v' (verb)
JJR → 'a' (adjective)
RB  → 'r' (adverb)
```

#### `guess_pos_from_wordnet(word)`
Checks which parts of speech WordNet recognizes for a given word.

#### `repair_pos(word, tag, context)`
The heart of our POS correction system. It:

- ✅ Fixes obviously wrong tags using WordNet as ground truth
- ✅ Detects when inflected forms (like "happens") are misidentified
- ✅ Preserves grammatical information (VBZ vs VBD vs VBG)
- ⚠️ **Never** downgrades "notifies" (VBZ) to generic "notify" (VB)

**Key Innovation**: We check both the inflected word AND its base form in WordNet, because WordNet only stores lemmas (base forms).

#### `get_core_words(sentence)`
Extracts meaningful content words:
1. Tokenize the sentence
2. Apply POS tagging
3. **Repair** incorrect tags
4. Filter out stopwords and punctuation
5. Keep only nouns, verbs, adjectives, and adverbs

In [ ]:
def penn_to_wn_pos(tag: str):
    """
    Convert Penn Treebank POS tags (NN, VBZ, JJ, RB...) into
    WordNet POS tags ('n', 'v', 'a', 'r').
    Return None if we don't care about that tag.
    Examples:
    - 'NN'   -> 'n'
    - 'VBZ'  -> 'v'
    - 'JJR'  -> 'a'
    - 'RB'   -> 'r'
    Args:
        tag: str, Penn Treebank POS tag
    Returns:
        str or None: WordNet POS tag or None
    """
    if tag.startswith('NN'):
        return 'n'
    if tag.startswith('VB'):
        return 'v'
    if tag.startswith('JJ'):
        return 'a'
    if tag.startswith('RB'):
        return 'r'
    return None


def guess_pos_from_wordnet(word: str):
    """
    Check which POS categories this word actually appears in (in WordNet).
    Useful for sanity-checking tags.
    Returns a dict with keys 'n', 'v', 'a', 'r' and boolean values.
    """
    return {
        'n': bool(wn.synsets(word, pos='n')),
        'v': bool(wn.synsets(word, pos='v')),
        'a': bool(wn.synsets(word, pos='a')),
        'r': bool(wn.synsets(word, pos='r')),
    }


def repair_pos(word, tag, context_tags):
    """
    Heuristic POS repair.

    Goals:
    - Fix clearly wrong POS tags (like 'quickly' being tagged as NN).
    - Do NOT destroy tense/person info on verbs (VBZ vs VBP vs VBD, etc.).
    - Check base forms when dealing with inflected words.

    Returns a repaired Penn tag.
    """
    
    def get_base_form(word):
        """Try to get base form of a word by removing common inflections."""
        if word.endswith('ies') and len(word) > 3:
            return word[:-3] + 'y'  # flies -> fly
        elif word.endswith('es') and len(word) > 3:
            if word.endswith(('shes', 'ches', 'xes', 'zes', 'sses')):
                return word[:-2]  # watches -> watch
            else:
                return word[:-1]  # likes -> like (might be wrong for goes->goe but we'll check)
        elif word.endswith('s') and len(word) > 2:
            return word[:-1]  # runs -> run
        elif word.endswith('ed') and len(word) > 3:
            if word.endswith(('ied')):
                return word[:-3] + 'y'  # tried -> try
            elif word[-3] == word[-4] and word[-4] not in 'aeiou':
                return word[:-3]  # stopped -> stop
            else:
                base = word[:-2] if word.endswith('eed') else word[:-1]
                return base if word[:-2] != word[:-1] else word[:-2]
        elif word.endswith('ing') and len(word) > 4:
            return word[:-3]  # running -> run (simplified)
        return word

    # 0. Adjectives ending with "y" that are actually adjectives
    if tag.startswith("NN") and word.endswith("y") and len(word) > 3:
        if wn.synsets(word, pos='a') and not wn.synsets(word, pos='n'): 
            return "JJ"
        
    # 1. JJS that's actually a noun
    if tag == "JJS":
        if wn.synsets(word, pos='n') and not wn.synsets(word, pos='a'):
            return "NN"

    # 2. Words ending with "ly" that are adverbs
    if tag.startswith("NN") and word.endswith("ly"):
        if wn.synsets(word, pos='r') and not wn.synsets(word, pos='n'):
            return "RB"

    # 3. Verb tagged as noun or noun tagged as verb - check base form too
    base = get_base_form(word)
    
    # 3a. If tagged as any verb form but WordNet says it's ONLY a noun
    if tag.startswith("VB"):
        is_noun = wn.synsets(word, pos='n') or wn.synsets(base, pos='n')
        is_verb = wn.synsets(word, pos='v') or wn.synsets(base, pos='v')
        if is_noun and not is_verb:
            return "NN"

    # 3b. If tagged as noun but WordNet says it's ONLY a verb
    #     This catches "happens" tagged as NNS
    if tag.startswith("NN"):
        is_noun = wn.synsets(word, pos='n') or wn.synsets(base, pos='n')
        is_verb = wn.synsets(word, pos='v') or wn.synsets(base, pos='v')
        
        if is_verb and not is_noun:
            # Now determine which verb form based on the word ending
            if word.endswith('s') and len(word) > 2 and not word.endswith('ss'):
                # Likely 3rd person singular: happens, notifies, runs
                if base != word and wn.synsets(base, pos='v'):
                    return "VBZ"
                else:
                    return "VB"
            elif word.endswith('ed'):
                return "VBD"
            elif word.endswith('ing'):
                return "VBG"
            else:
                return "VB"

    # 4. If tagged as VB but morphology suggests VBZ
    if tag == "VB":
        if word.endswith('s') and len(word) > 2 and not word.endswith('ss'):
            is_verb = wn.synsets(word, pos='v') or wn.synsets(base, pos='v')
            if is_verb and base != word:
                return "VBZ"

    # 5. Otherwise keep the original tag
    return tag



def get_core_words(sentence: str):
    """
    Extract 'core' content words from a sentence.
    Steps:
    1. Tokenize.
    2. POS-tag.
    3. Repair POS tags.
    4. Keep only content words (N, V, Adj, Adv) that are not stopwords/punctuation.

    Args:
        sentence: str, input sentence
    Returns:
        core_words: list[(word_lower, repaired_POS)]
        full_tagged: list[(original_token, original_POS)]
    """
    tokens = word_tokenize(sentence)
    tagged = pos_tag(tokens)

    # Repair tags FIRST
    repaired = []
    tags_only = [t for _, t in tagged]
    for i, (w, t) in enumerate(tagged):
        new_tag = repair_pos(w.lower(), t, tags_only)
        repaired.append((w.lower(), new_tag))

    # Filter using REPAIRED tags
    core = []
    for word, tag in repaired:
        if word in stop_en or word in punct:
            continue
        if tag.startswith(('NN', 'VB', 'JJ', 'RB')):
            core.append((word, tag))

    return core, tagged


## 🧠 Part 2: Sense Mapping

### The Goal

Assign each word its most appropriate meaning from WordNet.

### Our Approach

This is a simplified word-sense disambiguation (WSD):

1. Convert the repaired POS tag → WordNet POS
2. Query WordNet for all possible meanings (synsets)
3. Pick the **first synset** (usually the most common sense)

> **Note**: This is naive but effective for this assignment. Production systems use context-aware algorithms.

### Functions

#### `get_sense_map(core_words)`
Creates a dictionary mapping each word to its chosen synset.

#### `describe_word_senses(core_words, sense_map, fixed_tags)`
Prints human-readable output showing:
- The word
- Its POS tag
- The chosen synset
- The definition

**Example Output:**
```
praised (VBD): praise.v.01 -> express approval of
creativity (NN): creativity.n.01 -> the ability to create
```


In [ ]:
def get_sense_map(core_words):
    """
    Build:
    - sense_map: word -> chosen wn.Synset (or None)
    - fixed_tags: list of (word, repaired_tag)
    Steps:
    1. For each core word, convert repaired POS tag to WordNet POS tag.
    2. Lookup synsets in WordNet for that word and POS.
    3. Choose the first synset if any exist, else None.
    4. Store in sense_map and fixed_tags.
    Args:
        core_words: list[(word_lower, repaired_POS)]
    Returns:
        sense_map: dict[word_lower] -> wn.Synset or None
        fixed_tags: list[(word_lower, repaired_POS)]
    """
    sense_map = {}
    fixed_tags = []

    for (w, repaired_tag) in core_words:
        fixed_tags.append((w, repaired_tag))

        wn_pos = penn_to_wn_pos(repaired_tag)
        if wn_pos is None:
            sense_map[w] = None
            continue

        synsets = wn.synsets(w, pos=wn_pos)
        chosen = synsets[0] if synsets else None
        sense_map[w] = chosen

    return sense_map, fixed_tags


def describe_word_senses(core_words, sense_map, fixed_tags):
    """
    Human-readable dump of sense assignments.
    Prints: word (POS): synset.name() -> definition
    Args:
        core_words: list[(word_lower, repaired_POS)]
        sense_map: dict[word_lower] -> wn.Synset or None
        fixed_tags: list[(word_lower, repaired_POS)]
    Returns:
        None
    """
    for i, (w, tag) in enumerate(core_words):
        syn = sense_map[w]
        _, shown_tag = fixed_tags[i]

        if syn is None:
            print(f"{w} ({shown_tag}): NO SYNSET FOUND")
        else:
            print(f"{w} ({shown_tag}): {syn.name()} -> {syn.definition()}") # play.v.01 -> engage in activity for enjoyment and recreation rather than a serious or practical purpose.


## 🌲 Part 3: Finding Shared High-Level Concepts

### What Are We Looking For?

> "The lowest common hypernyms that capture the main theme of the sentence"

Think of WordNet as a tree structure:
```
entity.n.01 (very general)
  └─ physical_entity.n.01
      └─ object.n.01
          └─ artifact.n.01
              └─ structure.n.01
                  └─ building.n.01 (specific)
```

### Our Strategy

We focus on **nouns only** because their hypernym trees are most coherent:

1. Collect all noun synsets in the sentence
2. Walk up each noun's hypernym tree to find all ancestors
3. Count how often each ancestor appears across nouns
4. Keep ancestors that appear in ≥2 nouns (or 1 if only 1 noun exists)
5. Filter to keep only the **most specific** shared concepts

### Functions

#### `get_hypernym_chain(syn)`
Recursively collects all ancestor synsets up the tree.

#### `find_lowest_common_hypernyms(core_words, sense_map)`
Implements the full strategy above.

**Example**: For a sentence about "manager" and "employees":
```
✅ person.n.01 (shared by both)
❌ entity.n.01 (too general, excluded)
```

In [ ]:
from collections import Counter

def get_hypernym_chain(syn):
    """
    Return all hypernyms recursively upward for a synset, including itself.
    Used to build the ancestor sets for nouns.
    Args:
        syn (wn.Synset): The starting synset (e.g., 'dog.n.01')
    Returns:
        set of wn.Synset: All ancestor synsets, including the original one.
    """
    result = set()
    frontier = [syn]

    # Depth-first traversal upward through the hypernym hierarchy
    while frontier:
        current = frontier.pop()
        if current in result:
            continue  # Skip if we’ve already visited this synset
        result.add(current)
        # Add its immediate hypernyms (parents in the WordNet tree)
        frontier.extend(current.hypernyms())

    return result


def find_lowest_common_hypernyms(core_words, sense_map, min_support=0.5):
    """
    Among noun synsets in this sentence, find shared hypernyms.

    Strategy:
    1. Collect synsets for nouns only.
    2. For each noun synset, get its full hypernym chain (ancestors).
    3. Count how often each hypernym appears across all nouns.
    4. Keep hypernyms that are shared by >= threshold nouns.
       (Threshold = 2 nouns, unless there’s only 1 noun.)
    5. From those, discard any hypernym that has a *more specific*
       hypernym also in the set. Keep only the most specific (lowest) ones.

    Args:
        core_words (list[(str, str)]): List of (word_lower, repaired_POS)
        sense_map (dict): Mapping from word -> wn.Synset (or None)
        min_support (float): Minimum fraction of nouns that must share a hypernym
    Returns:
        list[wn.Synset]: Final list of lowest shared hypernyms
    """

    # Step 1: Collect noun synsets only (NN, NNS, etc.)
    noun_synsets = []
    for w, tag in core_words:
        if tag.startswith('NN'):
            syn = sense_map[w]
            if syn:
                noun_synsets.append(syn)

    # If no nouns → nothing to compare
    if not noun_synsets:
        return []

    # Step 2: Get all ancestor synsets for each noun
    chains = [get_hypernym_chain(s) for s in noun_synsets]

    # Step 3: Count how often each hypernym appears across all nouns
    counts = Counter() 
    for chain in chains:
        for h in chain:
            counts[h] += 1

    # -----------------------------
    # Step 4: Apply the threshold rule
    # -----------------------------
    # If the sentence has only 1 noun, allow that noun’s own hypernyms (threshold = 1)
    # Otherwise, require a hypernym to be shared by at least 2 different nouns
    threshold = 1 if len(noun_synsets) == 1 else 2

    # Keep only those hypernyms that meet or exceed the threshold count
    common_hypernyms = [h for h, c in counts.items() if c >= threshold]

    # If no shared hypernyms → nothing left
    if not common_hypernyms:
        return []

    # -----------------------------
    # Step 5: Keep only the most specific (lowest) shared hypernyms
    # -----------------------------
    def is_ancestor(a, b):
        """
        Return True if synset `a` is a (strict) ancestor of synset `b`
        in the WordNet hypernym tree. Example:
        is_ancestor('physical_entity.n.01', 'person.n.01') → True
        """
        return (a in get_hypernym_chain(b)) and (a != b)

    final_candidates = []
    for h in common_hypernyms:
        # Check if there exists a *more specific* hypernym (child) also shared
        lower_exists = any(is_ancestor(h, other) for other in common_hypernyms)
        if not lower_exists:
            # Keep only hypernyms that are not ancestors of any other shared one
            # → ensures we output the most specific shared concept
            final_candidates.append(h)


    return final_candidates


## ✨ Part 4: Paraphrasing by Synonym Substitution

### The Goal

Replace 3-5 content words with synonyms while:
- ✅ Preserving grammatical correctness
- ✅ Maintaining the original meaning
- ✅ Keeping proper verb tenses and noun plurals

### The Challenge

WordNet gives us base forms, but we need inflected forms:

| Original | Base from WordNet | We Need |
|----------|-------------------|---------|
| praised | praise | praised (VBD) |
| employees | employee | employees (NNS) |
| notifies | notify | notifies (VBZ) |

### Helper Functions

#### `pluralize(noun_base)`
Applies English pluralization rules:
- party → parties (consonant + y)
- box → boxes (ends in s/x/ch/sh/z)
- wolf → wolves (ends in f)
- cat → cats (default)

#### `valid_wordnet_form(token, wn_pos)`
Verifies that WordNet recognizes a word as valid for a given POS.

#### `pick_synonym_for_word(word, tag, syn)`
The core paraphrasing logic:

1. Get synonym candidates from the synset
2. Filter out:
   - ❌ Identical words
   - ❌ Multi-word phrases
   - ❌ Register shifts (green → greenish)
3. Apply morphology rules based on POS tag:

**For Verbs:**
```python
VBD  → add -ed (praised → commended)
VBG  → add -ing (discussing → proposing)
VBZ  → add -s/-es (notifies → advises)
VB   → keep base (achieve → accomplish)
```

**For Nouns:**
```python
NN   → keep singular (employee → worker)
NNS  → pluralize (employees → workers)
```

#### `detok(s)`
Cleans up tokenization artifacts:
- Removes spaces before punctuation
- Fixes parentheses spacing
- Capitalizes first letter

#### `paraphrase_sentence(original_sentence, core_words, sense_map, max_changes)`
Orchestrates the full paraphrasing process:
1. Find up to `max_changes` synonym replacements
2. Substitute them in the original tokens
3. Preserve capitalization of proper nouns
4. Detokenize the result


In [55]:
def pluralize(noun_base: str) -> str:
    """
    Naive pluralization rules for English nouns.

    We generate a plural form from a singular base word, using common English patterns.

    Rules / examples:
    1. word ends with consonant + "y"  → replace "y" with "ies"
       "party" -> "parties"
       "city"  -> "cities"
       BUT "key" -> "keys" (not consonant+y, so it falls through to default)

    2. word ends with "s", "x", "ch", "sh", "z", or "o" → add "es"
       "box"   -> "boxes"
       "class" -> "classes"
       "hero"  -> "heroes"

    3. word ends with "f" or "fe" → change to "ves"
       "wolf"  -> "wolves"
       "knife" -> "knives"

    4. default fallback → add "s"
       "cat"   -> "cats"
       "car"   -> "cars"

    This is intentionally simple and doesn't try to cover irregulars like "child" -> "children".
    """
    # consonant + y → ies  ("party" -> "parties")
    if noun_base.endswith("y") and len(noun_base) > 1 and noun_base[-2] not in "aeiou":
        return noun_base[:-1] + "ies"

    # s/x/ch/sh/z/o → add "es"  ("box" -> "boxes", "hero" -> "heroes")
    if noun_base.endswith(("s","x","ch","sh","z","o")):
        return noun_base + "es"

    # f / fe → ves  ("wolf" -> "wolves", "knife" -> "knives")
    if noun_base.endswith("f"):
        return noun_base[:-1] + "ves"
    if noun_base.endswith("fe"):
        return noun_base[:-2] + "ves"

    # default: just add "s"  ("cat" -> "cats")
    return noun_base + "s"


def valid_wordnet_form(token: str, wn_pos: str) -> bool:
    """
    Check if WordNet knows this token as a lemma for the given POS.

    Arguments:
      token  : the candidate word we might output (e.g. "achieved", "hero", "creative")
      wn_pos : one of 'n', 'v', 'a', 'r' (noun, verb, adjective, adverb)

    Returns:
      True  -> WordNet has at least one synset/lemma for that word with that POS.
               For example:
                 valid_wordnet_form("run", "v")   -> True (WordNet has 'run' as a verb)
                 valid_wordnet_form("run", "n")   -> True (WordNet also has 'run' as a noun)
                 valid_wordnet_form("asdfg", "n") -> False (nonsense)
      False -> no synset for that POS, or wn_pos is not in ('n','v','a','r').

    We use this to avoid outputting garbage synonyms WordNet doesn't actually treat
    as that POS.
    """
    if wn_pos not in ('n', 'v', 'a', 'r'):
        return False
    return bool(wn.synsets(token, pos=wn_pos))


def pick_synonym_for_word(word, tag, syn):
    """
    Pick a synonym-like replacement for `word` that:
    - matches part of speech category (noun, verb, adj, adv),
    - tries to keep morphology (plural nouns stay plural, VBD stays past tense, etc.),
    - avoids obviously weird drift (archaic / multiword).

    If we can't find a safe alternative, we just return the original `word`.

    Here are some example transformations we want to achieve:
    - Adjective (JJ): "creative" -> "innovative"
    - Adverb (RB): "quickly" -> "rapidly"
    - Noun singular (NN): "employee" -> "worker"
    - Noun plural (NNS): "employees" -> "workers"
    - Verb past tense (VBD): "praised" -> "commended"
    - Verb present participle (VBG): "discussing" -> "proposing"
    - Verb 3rd-person singular present (VBZ): "runs" -> "sprints"
    - Verb base form (VB): "achieve" -> "accomplish"

    Arguments:
      word : the original word token (lowercase)
      tag  : the POS tag for that word (Penn Treebank style, e.g. "NN", "VBD", "JJ", etc.)
      syn  : the chosen wn.Synset for that word (or None)

    Returns:
      str : a synonym-like replacement, or the original word if none found
    """
    if syn is None:
        return word

    candidates = [lemma.replace('_', ' ') for lemma in syn.lemma_names()]

    for cand in candidates:
        cand_clean = cand.strip()

        # 1. skip identical word ("team" -> "team")
        if cand_clean.lower() == word.lower():
            continue

        # 2. skip multiword phrases ("domestic cat")
        if " " in cand_clean:
            continue

        # --- ADJECTIVES: JJ / JJR / JJS ---
        if tag.startswith("JJ"):
            # e.g. "creative" -> "innovative"
            return cand_clean

        # --- ADVERBS: RB / RBR / RBS ---
        if tag.startswith("RB"):
            # e.g. "quickly" -> "rapidly"
            if valid_wordnet_form(cand_clean, 'r'):
                return cand_clean
            # Try adding -ly if base is an adjective
            if not cand_clean.endswith('ly'):
                ly_form = cand_clean + 'ly'
                if valid_wordnet_form(ly_form, 'r'):
                    return ly_form
            # if it's not a valid adverb, skip and try next candidate
            continue

        # --- NOUNS (singular): NN / NNP ---
        if tag in ("NN", "NNP"):
            # e.g. "employee" -> "worker"
            if valid_wordnet_form(cand_clean, 'n'):
                return cand_clean
            else:
                # bad noun candidate, try next
                continue

        # --- NOUNS (plural): NNS / NNPS ---
        if tag in ("NNS", "NNPS"):
            # e.g. "employees" -> "workers"
            plural_form = pluralize(cand_clean)
            # FIX 4: More lenient - check if base exists in WordNet
            if valid_wordnet_form(cand_clean, 'n'):
                return plural_form
            continue

        # --- VERBS: any VB* tag ---
        if tag.startswith("VB"):

            # Past / past participle
            if tag in ("VBD", "VBN"):
                if cand_clean.endswith("e"):
                    candidate = cand_clean + "d"
                elif cand_clean.endswith("y") and len(cand_clean) > 1 and cand_clean[-2] not in "aeiou":
                    candidate = cand_clean[:-1] + "ied"
                else:
                    candidate = cand_clean + "ed"

                # Check if base verb exists, not just inflected form
                if valid_wordnet_form(cand_clean, 'v'):  # <-- CHECKS BASE FORM
                    return candidate
                continue

            # Present participle / -ing (VBG)
            if tag == "VBG":
                # Example:
                #   word="discussing"
                #   cand_clean="propose"
                #   -> "proposing"
                if cand_clean.endswith("ing"):
                    candidate = cand_clean
                elif cand_clean.endswith("e"):
                    candidate = cand_clean[:-1] + "ing"     # "move" -> "moving"
                else:
                    candidate = cand_clean + "ing"          # "talk" -> "talking"

                # If WordNet recognizes that form as a verb at all, use it
                if valid_wordnet_form(candidate, 'v'):
                    return candidate
                else:
                    continue

            # Third-person singular present (VBZ)
            if tag == "VBZ":
                # Example:
                #   word="runs"
                #   cand_clean="run" -> "runs"
                #   word="flies"
                #   cand_clean="fly" -> "flies"
                if cand_clean.endswith(("s","x","ch","sh","z","o")):
                    candidate = cand_clean + "es"
                elif cand_clean.endswith("y") and len(cand_clean) > 1 and cand_clean[-2] not in "aeiou":
                    candidate = cand_clean[:-1] + "ies"
                else:
                    candidate = cand_clean + "s"

                if valid_wordnet_form(cand_clean, 'v'):
                    return candidate
                else:
                    # skip weird forms that aren't verbs
                    continue

            # Bare infinitive / non-3sg present (VB, VBP)
            if tag in ("VB", "VBP"):
                # Example:
                #   word="achieve" (VB)
                #   cand_clean="accomplish"
                #   -> "accomplish"
                if valid_wordnet_form(cand_clean, 'v'):
                    return cand_clean
                else:
                    continue

            # If it's some verb form we didn't explicitly handle, skip.
            continue

        # --- FALLBACK ---
        # If we got here, we didn't match any branch above.
        # We'll just return this candidate as a last resort.
        return cand_clean

    # If we exit the loop with nothing acceptable, keep the original.
    return word


def detok(s: str) -> str:
    """
    Detokenize a space-joined token string.

    Fixes:
    - Removes extra spaces before punctuation.
      "word ," -> "word,"
      "done !" -> "done!"

    - Fixes spaces inside parentheses.
      "( hello )" -> "(hello)"

    - Capitalizes first character of the final sentence.
      "the dog barked loudly ." -> "The dog barked loudly."
    """
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s*’\s*", "’", s)           # curly apostrophe
    s = re.sub(r"\s*'\s*", "'", s)           # straight apostrophe

    if len(s) > 0:
        s = s[0].upper() + s[1:]

    return s


def paraphrase_sentence(original_sentence: str, core_words, sense_map, max_changes=5):
    """
    Create a paraphrased version of `original_sentence` by swapping up to `max_changes`
    core content words with synonyms from WordNet.

    Steps:
    1. Tokenize the original sentence.
    2. For each (word, tag) in `core_words`, try to pick a synonym using `pick_synonym_for_word`.
       Example:
         ("praised", "VBD") might become "commended"
         ("employees", "NNS") might become "workers"
    3. Build a replacements dict like:
         {"praised": "commended", "employees": "workers"}
    4. Walk through tokens again and substitute if lowercase matches a key.
       Preserve capitalization of the first letter if the original token was capitalized.
    5. Detokenize to clean up punctuation spacing and capitalization.

    Arguments:
      original_sentence : the input sentence string
      core_words       : list of (word_lower, repaired_POS) for content words in the sentence
      sense_map        : dict mapping word_lower -> wn.Synset (or None)
      max_changes      : maximum number of words to change in the sentence
    Returns:
      A string that (ideally) keeps the same meaning but with superficial wording changes.
    """
    tokens = word_tokenize(original_sentence)

    # decide which words to replace
    replacements = {}
    changes = 0
    for w, tag in core_words:
        if changes >= max_changes:
            break
        syn = sense_map.get(w)
        alt = pick_synonym_for_word(w, tag, syn)
        if alt and alt != w:
            replacements[w] = alt
            changes += 1

    # rebuild using replacements
    new_tokens = []
    for tok in tokens:
        low = tok.lower()
        if low in replacements:
            new_word = replacements[low]

            # Preserve capitalization for tokens that start uppercase
            # Example:
            #   "Earth" -> "World" not "world"
            if tok[0].isupper():
                new_word = new_word[0].upper() + new_word[1:]

            new_tokens.append(new_word)
        else:
            new_tokens.append(tok)

    return detok(" ".join(new_tokens)), replacements


## 🚀 Part 5: Running the Pipeline

### `run_pipeline_on_paragraph(paragraph, max_changes=5)`

This function ties everything together:

```
Input Text
    ↓
[Sentence Tokenization]
    ↓
For each sentence:
    ├─ Extract core words
    ├─ Map to WordNet senses
    ├─ Find shared concepts
    ├─ Generate paraphrase
    └─ Display results
```
### Output Format

For each sentence, you'll see:

#### 🎯 Task 1: Shared High-Level Concepts
```
- person.n.01 -> a human being
- activity.n.01 -> any specific behavior
```

#### 🏷️ Task 2: Word Sense Assignments
```
manager (NN): director.n.01 -> someone who controls resources
praised (VBD): praise.v.01 -> express approval of
```

#### ✏️ Task 3: Paraphrase Generation
```
Replacements made:
  manager  ->  director
  praised  ->  commended
  
Original: The manager praised the team.
Paraphrased: The director commended the team.
```

In [56]:
def run_pipeline_on_paragraph(paragraph: str, max_changes=5):
    sentences = sent_tokenize(paragraph)

    for sentence in sentences:
        print("====================================================")
        print("Original Sentence:")
        print(sentence)

        # Step 1: core words
        core_words, tagged_full = get_core_words(sentence)

        # Step 2: sense mapping
        sense_map, fixed_tags = get_sense_map(core_words)

        # Step 3: shared high-level hypernyms
        lch = find_lowest_common_hypernyms(core_words, sense_map)

        print("\n[Task 1] Shared high-level concepts (lowest common hypernyms):")
        if not lch:
            print("  None found.")
        else:
            for syn in lch:
                print(f"  - {syn.name()} -> {syn.definition()}")

        print("\n[Task 2] Word sense assignments:")
        describe_word_senses(core_words, sense_map, fixed_tags)

        # Step 4: paraphrase
        paraphrased, replacements = paraphrase_sentence(
            sentence,
            core_words,
            sense_map,
            max_changes=max_changes
        )

        print("\n[Task 3]")
        print("Replacements made:")
        if not replacements:
            print("  None.")
        else:
            for orig, new in replacements.items():
                print(f"  {orig}  ->  {new}")
        print("\nOriginal Sentence:")
        print(sentence)
        print("Paraphrased Sentence:")
        print(paraphrased)
        print("====================================================\n")

nltk.pos_tag(["happens"])


[('happens', 'NNS')]

## 🧪 Part 6: Demo Example

We test on a multi-sentence paragraph to demonstrate all capabilities:

- Handles different sentence structures
- Works with various POS tags (VBD, VBZ, VBG, NNS, etc.)
- Finds appropriate synonyms while preserving grammar
- Identifies conceptual themes across complex sentences

---
## 🔧 Technical Notes

### Known Limitations

1. **POS Tagger Accuracy**: NLTK's tagger isn't perfect, especially with:
   - Verbs in complex sentence structures
   - Words that can be multiple POS (e.g., "flag" as noun or verb)

2. **Synonym Quality**: WordNet synonyms aren't always perfect substitutes:
   - "data → informations" (grammatically incorrect)
   - Some synonyms are too formal or archaic

3. **Morphology Rules**: Our inflection rules are heuristic-based and won't handle:
   - Irregular verbs (go → went)
   - Irregular plurals (child → children)

In [58]:
example_paragraph = (
"""The engineer adjusted the small drone before launching it into the cloudy sky.
It hovered quietly over the busy street, capturing images of moving cars and flashing lights.
Later, the team analyzed the collected data to improve the drone's flight stability.""")

run_pipeline_on_paragraph(example_paragraph, max_changes=5)


Original Sentence:
The engineer adjusted the small drone before launching it into the cloudy sky.

[Task 1] Shared high-level concepts (lowest common hypernyms):
  - organism.n.01 -> a living thing that has (or can develop) the ability to act or function independently

[Task 2] Word sense assignments:
engineer (NN): engineer.n.01 -> a person who uses scientific knowledge to solve practical problems
adjusted (VBD): adjust.v.01 -> alter or regulate so as to achieve accuracy or conform to a standard
small (JJ): small.a.01 -> limited or below average in number or quantity or magnitude or extent
drone (NN): drone.n.01 -> stingless male bee in a colony of social bees (especially honeybees) whose sole function is to mate with the queen
launching (VBG): establish.v.01 -> set up or found
cloudy (JJ): cloudy.s.01 -> lacking definite form or limits; - H.T.Moore
sky (NN): sky.n.01 -> the atmosphere and outer space as viewed from the earth

[Task 3]
Replacements made:
  engineer  ->  technologist
 